# Spectrum Kernel GPR
- Trying to fix GPR performance by using a sum of SHOTerms to approximate the Gonzalez kernel
- If this is collapsing to boundaries again, try using the variance limitation with softmax.

- Writing a generalised function to fir the GPR using a spectrum kernal with K terms.
- Spectrum kernel here is a sum of SHO terms which, in the underdamped Q>0.5 regime, is equivalent to a sum of K quasi-normal terms.
- Each SHO has:
    - Magnitude defined by the sigma.
    - Period defined by rho.
    - Quality factor Q defining the exp decay rate; constrained to > 0.5 for underdamped regime.

In [ ]:
import pandas as pd
from astropy.time import Time
import matplotlib.pyplot as plt
import celerite2
import numpy as np
from celerite2 import terms
from scipy.optimize import minimize

def set_params(log_params, k, gp):
    # k = len(log_params) // 3
    params = np.exp(log_params)
    sigmas = params[0:k]
    rhos = params[k:2k]
    qs = params[2k::]
    
    gp.kernel = terms.SHOTerm(sigmas[0], rhos[0], qs[0])
    for k_idx in range(1,k):
        gp.kernel += terms.SHOTerm(sigmas[k_idx], rhos[k_idx], qs[k_idx])
    return gp

def NLL(log_params, gp, y):
    '''
    Calculates the NLL of a set of parameters for a local gp
    '''
    gp = set_params(log_params, gp)
    gp.recompute(quiet=True)
    return -gp.log_likelihood(y)

def fit_spec_gpr(k=2, plot = True):
    '''
    k is at minimum 1
    '''
    # Create the initial conditions as required by the k value
    # Create sigmas, then rhos, then qs
    # Take the log to create initial guesses
    # Create the bounds
    # Create the initial kernel
    # Define the gp
    gp = celerite2.GaussianProcess(kernel, mean=train_mean)
    gp.compute(train_df['year'], yerr=train_yerr)
    # Train the gp
    gp_res = minimize(NLL, initial_guess, args=(gp, train_df["sind"].to_numpy()), method="L-BFGS-B", bounds=bounds)
    gp = set_params(gp_res.x, gp)
    gp.recompute()
    # Print the opt results
    best = np.exp(gp_res.x)
    blo = np.exp([b[0] for b in bounds])
    bhi = np.exp([b[1] for b in bounds])
    print(gp_res.success, gp_res.message)
    print(f"RotationTerm: sigma={best[0]:.5f} [{blo[0]:.4f},{bhi[0]:.4f}]  period={best[1]:.3f} [{blo[1]:.2f},{bhi[1]:.2f}] yr  Q0={best[2]:.3f} [{blo[2]:.2f},{bhi[2]:.2f}]  dQ={best[3]:.4f} [{blo[3]:.4f},{bhi[3]:.2f}]  f={best[4]:.3f} [{blo[4]:.3f},{bhi[4]:.3f}]")
    print(f"SHOTerm:      sigma={best[5]:.5f} [{blo[5]:.4f},{bhi[5]:.4f}]  period={best[6]:.3f} [{blo[6]:.1f},{bhi[6]:.1f}] yr  Q={best[7]:.4f}   [{blo[7]:.3f},{bhi[7]:.3f}]")
    # Predict forewards
    t_pred = valid_df['year']
    mu, cov = gp.predict(train_df['sind'], t = t_pred, return_var = True) # lt indicates long term component from SHOTerm
    sigma = np.sqrt(cov)
    # Plot
    if plot:
        results_shrt = pd.DataFrame({
        'forecast' : mu,
        'lower'    : mu - sigma,
        'upper'    : mu + sigma,
        }, index=valid_df.index)

        plot_predictions(
        trainingset=train_df,
        validset=valid_df,
        results=results
        )
    

    
